# A6 · Regression

*Path A: Write It Yourself, notebook 6 of 6 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Fit a linear regression and read its output
- Adjust for other variables
- Fit a logistic regression for a yes/no outcome
- Turn logistic regression results into odds ratios

**Before this notebook:** [A5: Statistical Tests](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/A5_Statistical_Tests.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Setup

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal. This notebook also loads `broom`, a package that turns model results into tidy tables.

In [ ]:
library(tidyverse)
library(broom)

study_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/final_dataset2.csv")

## 2. Simple linear regression

**Question:** how does systolic blood pressure change with age?

In notebook A4 you saw in a scatter plot that blood pressure rises with age. Linear regression puts a number on that: it finds the straight line that best fits the points.

The function is `lm()`, for "linear model". It uses the same formula style as the t-test: outcome, tilde, predictor.

In [ ]:
lm_model <- lm(SBP ~ age, data = study_df)
summary(lm_model)

The output is long. Look at the **Coefficients** table.

- The row **`age`**, column **`Estimate`**: 0.506. This is the slope. For each additional year of age, blood pressure is higher by about 0.5 mmHg on average.
- The same row, column **`Pr(>|t|)`**: the p-value for the slope. `4.75e-16` is scientific notation, as in notebook A5: a decimal point, 15 zeros, and then 475. Extremely small.
- **`(Intercept)`**: where the line crosses the vertical axis, at age 0. Rarely interesting in itself.
- Near the bottom, **`Multiple R-squared`**: 0.51. Age accounts for about half of the variation in blood pressure in this data.

### A tidier table

`tidy()` from the broom package shows the coefficients as a dataframe, which is easier to read and to use in later steps.

In [ ]:
tidy(lm_model)

### Seeing the line

Add `geom_smooth(method = "lm")` to a scatter plot to draw the fitted line. The shaded band shows the uncertainty around it.

In [ ]:
ggplot(study_df, aes(x = age, y = SBP)) +
  geom_point(color = "darkblue", alpha = 0.5) +
  geom_smooth(method = "lm", color = "red") +
  labs(
    title = "Systolic Blood Pressure by Age",
    x = "Age (years)",
    y = "SBP (mmHg)"
  ) +
  theme_minimal()

(`alpha = 0.5` makes the points half transparent. The message about `formula = 'y ~ x'` is normal, and the warnings about 5 removed rows are the 5 people with missing blood pressure.)

## 3. Multiple linear regression

In medical research you usually need to account for other variables: is the relationship still there after **adjusting** for sex, for example? To adjust for a variable, add it to the formula with a plus sign.

In [ ]:
lm_adjusted <- lm(SBP ~ age + sex_at_birth, data = study_df)
tidy(lm_adjusted)

- **`age`**: 0.504. After adjusting for sex, the slope for age has barely changed.
- **`sex_at_birthMale`** and **`sex_at_birthOther`**: for a category, R compares each group with a **reference group**, which is the first one alphabetically. Here that is Female. So males average about 1.3 mmHg higher than females of the same age, and that difference is not statistically significant (p = 0.55).

**Your turn 1.** Fit a linear regression of `SBP` on `age` and `exposure_status`, and show the tidy table. After adjusting for age, is drug exposure associated with blood pressure?

*Expected result: the estimate for exposure_status is about -2.03 with p-value 0.36*

In [ ]:
# Your code here

## 4. Logistic regression

**Question:** is drug exposure associated with the condition, after adjusting for age and sex?

The outcome here, `condition_status`, is yes or no (1 or 0). A straight line is the wrong tool for a yes/no outcome: it can predict values below 0 or above 1, which make no sense as probabilities.

**Logistic regression** is the standard method for a yes/no outcome. The key ideas:

- The **odds** of an outcome are the probability that it happens divided by the probability that it does not. A probability of 0.2 is odds of 0.2 / 0.8 = 0.25.
- Instead of modeling the outcome directly, logistic regression models the **logarithm of the odds**.
- So the coefficient it reports for each predictor is a **log odds ratio**.
- To get an **odds ratio**, you reverse the logarithm with the exponential function, `exp()`.

An odds ratio of 1 means no association. Above 1 means higher odds of the outcome; below 1 means lower odds.

The function is `glm()`, for "generalized linear model". `family = "binomial"` is what makes it a logistic regression.

In [ ]:
logistic_model <- glm(
  condition_status ~ exposure_status + age + sex_at_birth,
  data = study_df,
  family = "binomial"
)
tidy(logistic_model)

The estimate for `exposure_status` is about 2.16. **This is a log odds ratio, not an odds ratio.** Reporting it as "the odds ratio is 2.16" is a common and serious mistake. Convert it:

In [ ]:
exp(2.157)

The odds ratio is about 8.6. `tidy()` can do the conversion for every coefficient, without rounding, and add confidence intervals:

In [ ]:
tidy(logistic_model, exponentiate = TRUE, conf.int = TRUE)

Now the `estimate` column holds odds ratios, and `conf.low` and `conf.high` give the 95% confidence interval. (Two details: the `(Intercept)` row is not an odds ratio and is rarely of interest, and the `std.error` column is not converted.)

**How to say it:** after adjusting for age and sex, people exposed to the drug had 8.6 times the odds of the condition compared with unexposed people (95% confidence interval 3.3 to 24.3). The interval does not include 1, which agrees with the very small p-value.

**How not to say it:** "8.6 times as likely". An odds ratio is not a ratio of probabilities. When the outcome is common, as it is here (60% of exposed people and 14% of unexposed people have the condition), the odds ratio is much larger than the ratio of the two probabilities, which is about 4.

Remember that these data are simulated. The result illustrates the method; it says nothing about real medications.

**Your turn 2.** Fit an **unadjusted** logistic regression, with `exposure_status` as the only predictor of `condition_status`. Show the odds ratio and its confidence interval. How does it compare with the adjusted odds ratio of 8.64?

*Expected result: odds ratio 9.33 (95% confidence interval 3.63 to 25.9)*

In [ ]:
# Your code here

## 5. Before you trust a model

Regression models rest on assumptions about the data. Before drawing conclusions from one, check those assumptions with **model diagnostics**. That is beyond this notebook; see your statistics text or the [Additional Resources](https://icohr-lab.github.io/aou-workshop/resources.html) page.

## 6. Good to know for your own project

- **Objects live in memory, files live on disk.** `study_df` exists only while your session is running. If the session restarts, run your notebook again from the top. Use `ls()` to list the objects currently in memory.
- **Where am I?** `getwd()` shows the folder R is working in, which matters when you read or save files by name.
- **Dates read as text.** In *All of Us* data, dates sometimes arrive as text such as "2020-03-15 00:00:00 UTC". Convert them with `as.Date()`, which you met in notebook F2.
- **Write code that runs top to bottom.** Anyone, including you in six months, should be able to run your notebook from the first cell to the last and get the same result.

In [ ]:
ls()      # objects in memory
getwd()   # current working folder

---
## Recap

| Code | What it does |
|---|---|
| `lm(y ~ x, data = df)` | Linear regression |
| `lm(y ~ x + z, data = df)` | Linear regression adjusting for z |
| `glm(y ~ x, data = df, family = "binomial")` | Logistic regression (y is 0 or 1) |
| `summary(model)` | Full model output |
| `tidy(model)` | Coefficients as a tidy table |
| `tidy(model, exponentiate = TRUE, conf.int = TRUE)` | Odds ratios with confidence intervals |
| `exp(x)` | Exponential: converts a log odds ratio to an odds ratio |
| `geom_smooth(method = "lm")` | Add a fitted line to a scatter plot |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Your turn 1**

No evidence of an association: exposed people average about 2 mmHg lower at the same age, a difference that could easily be due to chance.

```r
lm_exposure <- lm(SBP ~ age + exposure_status, data = study_df)
tidy(lm_exposure)
```

**Your turn 2**

Adjusting for age and sex changed the odds ratio only slightly, from 9.33 to 8.64.

```r
unadjusted_model <- glm(
  condition_status ~ exposure_status,
  data = study_df,
  family = "binomial"
)
tidy(unadjusted_model, exponentiate = TRUE, conf.int = TRUE)
```

---
## Next

You have finished Path A. You can now clean, combine, explore, and analyze a dataset in R.

The next step is to do the same with real data in the *All of Us* Researcher Workbench. See the [workshop website](https://icohr-lab.github.io/aou-workshop/) for how to get started there.